# RFM segmentation: từ điểm số tới bảng hành động cho CRM

Head of CRM có ngân sách cho ba chiến dịch quý tới và hỏi: **"Nên nói chuyện với nhóm khách nào, nói gì?"**. RFM là
câu trả lời mà đội CRM dùng được ngay: mỗi khách có ba con số tại ngày chốt (Recency, Frequency, Monetary), đổi
thành điểm 1-5, rồi một bảng luật gom tổ hợp điểm thành segment có tên.

Bài này dùng **luật nghiệp vụ** (cách CRM thật dùng), không phải k-means (k-means trên RFM quay lại ở module ML cho
DA). Dữ liệu: Online Retail II (UCI, CC BY 4.0), sheet `Year 2010-2011`, dòng hợp lệ theo định nghĩa dùng chung,
ngày chốt 10/12/2011 (ngày sau giao dịch cuối).

**Cách làm bài**
- Cell chấm điểm chỉ định nghĩa hàm hoặc hằng. Điền vào chỗ `# TODO`; hàm không được sửa dữ liệu đầu vào.
- Cell gắn tag `explore` đọc dữ liệu thật và vẽ; khi chấm chúng bị bỏ qua.
- Chấm: `uv run pytest tracks/data-analyst/modules/12-product-customer-analytics/exercises/intermediate/01-rfm-segments -q`

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

In [ ]:
# Viết sẵn, không cần sửa: đọc Online Retail II (cache parquet) và lọc dòng hợp lệ theo định nghĩa dùng chung
import tempfile
import uuid
from pathlib import Path

CACHE_DIR = Path(tempfile.gettempdir()) / "da-cache"  # cache trong thư mục tạm, không ghi vào data/raw


def read_sheet(sheet: str) -> pd.DataFrame:
    """Một sheet Online Retail II. Lần đầu đọc xlsx (30-90 giây) rồi cache parquet theo mtime và cỡ tệp."""
    root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
    xlsx = root / "data" / "raw" / "online-retail-ii" / "online_retail_II.xlsx"
    if not xlsx.exists():
        raise FileNotFoundError("Chưa có dữ liệu. Chạy: uv run python data/download_dataset.py online-retail-ii")
    stat = xlsx.stat()
    cache = CACHE_DIR / f"online_retail_II-{sheet.replace(' ', '_')}-{stat.st_mtime_ns}-{stat.st_size}.parquet"
    if cache.exists():
        try:
            return pd.read_parquet(cache)
        except (OSError, ValueError):  # cache hỏng: đọc lại
            pass
    tx = pd.read_excel(xlsx, sheet_name=sheet)
    tx = tx.assign(Invoice=tx["Invoice"].astype(str), StockCode=tx["StockCode"].astype(str),
                   Description=tx["Description"].astype("string"))
    CACHE_DIR.mkdir(parents=True, exist_ok=True)
    tmp = cache.with_name(f"{cache.name}.{uuid.uuid4().hex}.tmp")
    tx.to_parquet(tmp, index=False)
    tmp.replace(cache)
    return tx


def clean_sales(tx: pd.DataFrame) -> pd.DataFrame:
    """Bỏ hoá đơn huỷ (C...), Quantity <= 0, Price <= 0; thêm revenue = Quantity * Price."""
    s = tx.assign(Invoice=tx["Invoice"].astype(str))
    s = s[~s.Invoice.str.startswith("C") & (s.Quantity > 0) & (s.Price > 0)].copy()
    s["revenue"] = s.Quantity * s.Price
    return s

## 1. Ba con số và điểm 1-5

`rfm_table(sales, snapshot) -> DataFrame`: bỏ dòng không có `Customer ID`; index là mã khách, ba cột theo thứ tự:

- `recency`: số ngày (số nguyên) từ **ngày** mua cuối (bỏ phần giờ, `dt.normalize()`) tới `snapshot`;
- `frequency`: số hoá đơn khác nhau (`Invoice`);
- `monetary`: tổng `revenue`.

`score_rfm(rfm, r_edges=(14, 30, 90, 180), f_edges=(1, 2, 4, 9)) -> DataFrame`: bản sao của `rfm` thêm ba cột `R`,
`F`, `M` (số nguyên 1-5, 5 là tốt nhất):

| Điểm | R (recency, ngày) | F (số đơn) | M |
| --- | --- | --- | --- |
| 5 | ≤ 14 | ≥ 10 | quintile cao nhất |
| 4 | 15-30 | 5-9 | |
| 3 | 31-90 | 3-4 | |
| 2 | 91-180 | 2 | |
| 1 | > 180 | 1 | quintile thấp nhất |

R và F dùng **ngưỡng nghiệp vụ** (cột `r_edges`, `f_edges` là mốc trên của từng mức, "≤ mốc"). M dùng **quintile**:
xếp hạng `monetary` với `method="first"` (giá trị trùng vẫn chia đều) rồi chia 5 nhóm bằng nhau.

In [ ]:
def rfm_table(sales: pd.DataFrame, snapshot: pd.Timestamp) -> pd.DataFrame:
    """Index khách; recency (ngày), frequency (số hoá đơn), monetary (tổng revenue)."""
    g = sales[sales["Customer ID"].notna()].groupby("Customer ID")
    return pd.DataFrame({
        "recency": (snapshot - g.InvoiceDate.max().dt.normalize()).dt.days,
        "frequency": g.Invoice.nunique(),
        "monetary": g.revenue.sum(),
    })


def score_rfm(rfm: pd.DataFrame, r_edges=(14, 30, 90, 180), f_edges=(1, 2, 4, 9)) -> pd.DataFrame:
    """Thêm R, F (ngưỡng) và M (quintile theo hạng) vào bản sao của rfm."""
    out = rfm.copy()
    out["R"] = 5 - np.searchsorted(list(r_edges), out.recency.to_numpy(), side="left")
    out["F"] = 1 + np.searchsorted(list(f_edges), out.frequency.to_numpy(), side="left")
    out["M"] = pd.qcut(out.monetary.rank(method="first"), 5, labels=[1, 2, 3, 4, 5]).astype(int)
    return out

## 2. Từ điểm tới segment

`SEGMENTS` là bảng luật dùng R và F (lưới phổ biến của CRM; M dùng để ưu tiên **trong** segment). Mỗi luật là
`(tên, các điểm R, các điểm F)`; **luật đầu tiên khớp thắng**. Bảng dưới đã phủ đủ 25 ô R × F, đừng sửa: test dùng nó.

`assign_segment(scores, rules) -> Series`: tên segment cho từng dòng của `scores` (cùng index), tên Series
`segment`; không luật nào khớp thì `"other"`.

`segment_summary(rfm, segment) -> DataFrame`: theo segment, cột theo thứ tự `customers`, `revenue` (tổng
`monetary`), `median_recency`, `median_frequency`, `customer_share`, `revenue_share`; sắp theo `revenue` giảm dần.

In [ ]:
SEGMENTS = [  # (tên, điểm R, điểm F): luật đầu tiên khớp thắng
    ("champions", [5], [4, 5]), ("loyal", [3, 4], [4, 5]), ("potential_loyalist", [4, 5], [2, 3]),
    ("new", [5], [1]), ("promising", [4], [1]), ("need_attention", [3], [3]), ("about_to_sleep", [3], [1, 2]),
    ("cant_lose", [1, 2], [5]), ("at_risk", [1, 2], [3, 4]), ("hibernating", [1, 2], [1, 2]),
]


def assign_segment(scores: pd.DataFrame, rules: list) -> pd.Series:
    """Tên segment theo luật đầu tiên khớp (R, F); không khớp -> "other"."""
    names = [next((name for name, rs, fs in rules if r in rs and f in fs), "other")
             for r, f in zip(scores.R, scores.F)]
    return pd.Series(names, index=scores.index, name="segment")


def segment_summary(rfm: pd.DataFrame, segment: pd.Series) -> pd.DataFrame:
    """customers, revenue, median_recency, median_frequency, customer_share, revenue_share; sắp theo revenue."""
    out = rfm.assign(segment=segment).groupby("segment").agg(
        customers=("monetary", "size"), revenue=("monetary", "sum"),
        median_recency=("recency", "median"), median_frequency=("frequency", "median"))
    out["customer_share"] = out.customers / out.customers.sum()
    out["revenue_share"] = out.revenue / out.revenue.sum()
    return out.sort_values("revenue", ascending=False)

In [ ]:
# explore: RFM trên sheet 2010-2011
sales = clean_sales(read_sheet("Year 2010-2011"))
snapshot = sales.InvoiceDate.max().normalize() + pd.Timedelta(days=1)
rfm = rfm_table(sales, snapshot)
scores = score_rfm(rfm)
segment = assign_segment(scores, SEGMENTS)
summary = segment_summary(rfm, segment)
print(f"{len(rfm):,} khách, snapshot {snapshot:%Y-%m-%d}, doanh thu {rfm.monetary.sum():,.0f} GBP")
print(rfm.describe().round(1).to_string())
print("R:", scores.R.value_counts().sort_index().to_dict(), "| F:", scores.F.value_counts().sort_index().to_dict())
print(summary.round(3).to_string())

In [ ]:
# explore: lưới R × F (số khách), và % khách vs % doanh thu theo segment
grid = scores.pivot_table(index="R", columns="F", values="M", aggfunc="size", fill_value=0).sort_index(ascending=False)
fig, (a, b) = plt.subplots(1, 2, figsize=(13, 4.5), gridspec_kw={"width_ratios": [1, 1.4]})
im = a.imshow(grid.to_numpy(), cmap="Blues")
for i in range(grid.shape[0]):
    for j in range(grid.shape[1]):
        a.annotate(f"{grid.iat[i, j]}", (j, i), ha="center", va="center", fontsize=8)
a.set_xticks(range(5), grid.columns)
a.set_yticks(range(5), grid.index)
a.set_xlabel("F (số đơn)")
a.set_ylabel("R (gần đây)")
a.set_title("Số khách theo ô R × F", loc="left")

order = summary.index[::-1]
y = np.arange(len(order))
b.barh(y + 0.2, summary.loc[order, "customer_share"] * 100, height=0.4, color="#9ecae9", label="% khách")
b.barh(y - 0.2, summary.loc[order, "revenue_share"] * 100, height=0.4, color="#4c78a8", label="% doanh thu")
b.set_yticks(y, order)
b.set_xlabel("%")
b.legend(frameon=False)
b.spines[["top", "right"]].set_visible(False)
champ = summary.loc["champions"]
b.set_title(f"Champions: {champ.customer_share:.0%} khách, {champ.revenue_share:.0%} doanh thu", loc="left")
fig.tight_layout()
plt.show()

In [ ]:
# explore: ba câu hỏi kiểm định nghĩa trước khi gửi bảng cho CRM
# 1) "new" có thật là khách mới? Kiểm lịch sử ở sheet 2009-2010
history = clean_sales(read_sheet("Year 2009-2010"))
before = set(history.loc[history.InvoiceDate < "2010-12-01", "Customer ID"].dropna())
new_ids = segment.index[segment == "new"]
print(f"'new': {len(new_ids)} khách, {len(set(new_ids) & before)} đã mua trước 12/2010")
# 2) at_risk: bao nhiêu khách giá trị cao (M >= 4) đáng gọi điện?
at_risk = scores[segment == "at_risk"]
print(f"at_risk: {len(at_risk)} khách, {(at_risk.M >= 4).sum()} khách M >= 4, "
      f"doanh thu {rfm.monetary[segment == 'at_risk'].sum():,.0f} GBP")
# 3) Monetary bị khách sỉ kéo tới đâu?
top10 = rfm.monetary.nlargest(10)
print(f"10 khách lớn nhất: {top10.sum() / rfm.monetary.sum():.1%} doanh thu năm; lớn nhất {top10.iloc[0]:,.0f} GBP")
print("ngưỡng quintile M (GBP):", rfm.monetary.quantile([0.2, 0.4, 0.6, 0.8]).round(0).to_dict())

## Đọc bảng, rồi làm trắc nghiệm

1. Champions và loyal cộng lại chiếm bao nhiêu % khách và bao nhiêu % doanh thu? Nhóm đông nhất là nhóm nào?
2. Với mỗi segment trong bảng hành động của lý thuyết (phần 5), viết **một** hành động và **một** metric đo nó.
3. "New" ở đây nghĩa là gì, theo đúng những gì dữ liệu biết? Nên đổi tên segment thế nào?
4. Nếu chạy lại RFM tháng sau, khách nào có thể đổi segment dù không đổi hành vi? (gợi ý: điểm nào dùng quintile)
5. Head of CRM muốn gửi mã giảm 20% cho cả nhóm at_risk. Bạn đề xuất gì thay vào đó, và đo kết quả thế nào?

Xong thì mở app, làm phần **trắc nghiệm** của bài.